# Trend-Following Backtest on Real Prices

**Trading · Intermediate · ~20 min · pandas + yfinance · needs an internet connection**

Test the classic moving-average crossover on ten years of real daily prices from Yahoo Finance (through the `yfinance` package): hold the asset while its 50-day average is above its 200-day average, sit in cash otherwise. The backtest is careful about the two things that make most backtests lie. It only trades at the **next day's open** after a signal, so it never uses a price it could not have known, and it charges a cost on every trade. The result is compared with simply buying and holding. The notebook needs an internet connection and writes nothing to disk.

**What you will see**
- The price with both moving averages and every entry and exit marked
- The equity curve of the strategy vs buy & hold, with their drawdowns
- A statistics table for both: total return, CAGR, volatility, Sharpe, max drawdown, number of trades, time in the market
- A table of every trade: entry date, exit date, days held and return

## 1. Parameters

`COST_BPS` is charged on every entry and every exit, in basis points of the amount traded (1 bp = 0.01%), to cover commission, bid-ask spread and slippage. 5 bp per trade means 0.10% per round trip. Try 20 or 50 to see how quickly costs eat a strategy that trades often.

In [ ]:
# ── Parameters: edit these, then Run All (Ctrl+Shift+Enter) ──────────
SYMBOL = "SPY"        # any Yahoo Finance symbol: "QQQ", "AAPL", "GLD", "BTC-USD" ...
PERIOD = "10y"        # history to download: "5y", "10y", "20y" or "max"
FAST = 50             # fast moving average (trading days)
SLOW = 200            # slow moving average (trading days)
COST_BPS = 5          # cost per trade (each entry and each exit), basis points of the trade value
START_VALUE = 10_000  # dollars at the start, for the equity curve

## 2. Imports and chart style

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

# One consistent, quiet chart style for every figure in this notebook
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
                                         "#e87ba4", "#008300", "#6250d6", "#e34948"]),
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold",
    "lines.linewidth": 1.6, "legend.frameon": False,
})

## 3. Download prices

We need the **open** as well as the close, because trades happen at the open. With `auto_adjust=True` both are adjusted for splits and dividends, so the returns include dividends.

In [ ]:
data = yf.download(SYMBOL, period=PERIOD, auto_adjust=True, progress=False)
if data.empty:
    raise ValueError(f"No price data for {SYMBOL} - check the symbol or your connection")

px = data.droplevel("Ticker", axis=1)[["Open", "Close"]].dropna()
if len(px) <= SLOW + 20:
    raise ValueError(f"Only {len(px)} days of data - use a PERIOD well beyond {SLOW} trading days")

print(f"{SYMBOL}: {len(px)} trading days, {px.index[0]:%Y-%m-%d} to {px.index[-1]:%Y-%m-%d}")
px.tail(5).round(2)

## 4. Signal and position, with no look-ahead

The timeline for every trading day:

1. **At today's close** both moving averages are computed from closes up to and including today. `signal = 1` if fast > slow, else 0.
2. **At tomorrow's open** we trade to match that signal, so the position held during day *t* is the signal from day *t−1*: `position = signal.shift(1)`.
3. The overnight move (yesterday's close to today's open) belongs to the position held *yesterday*; the open-to-close move belongs to today's position.

Applying today's signal to today's return would let the strategy react to a close it has not seen yet. That is *look-ahead bias*, and it flatters crossover backtests a lot. The cost is charged on each day the position changes. Buy & hold pays the same cost once, when it buys on the first day.

In [ ]:
px["sma_fast"] = px["Close"].rolling(FAST).mean()
px["sma_slow"] = px["Close"].rolling(SLOW).mean()
px = px.dropna().copy()                                         # start once the slow average exists
px["signal"] = (px["sma_fast"] > px["sma_slow"]).astype(int)    # known at today's close
px["position"] = px["signal"].shift(1).fillna(0).astype(int)    # held from today's open

cost = COST_BPS / 10_000
gap = (px["Open"] / px["Close"].shift(1) - 1).fillna(0)         # yesterday's close -> today's open
intraday = px["Close"] / px["Open"] - 1                          # today's open -> today's close
held_overnight = px["position"].shift(1).fillna(0)              # what we carried into this morning
trade = px["position"].diff().abs().fillna(0)                   # 1 on days we buy or sell at the open

px["strategy"] = (1 + held_overnight * gap) * (1 + px["position"] * intraday) * (1 - trade * cost) - 1
px["buy_hold"] = px["Close"].pct_change().fillna(-cost)         # day 1: buy at the close, pay the cost

print(f"Position changes: {int(trade.sum())}; in the market on {px['position'].mean():.0%} of days")
px[["Close", "sma_fast", "sma_slow", "signal", "position", "strategy"]].tail(8).round(4)

## 5. Price, averages and trades

Green triangles are buys and red triangles are sells, drawn at the open price where the trade happened, one day after the averages crossed.

In [ ]:
entries = px.index[(trade == 1) & (px["position"] == 1)]
exits = px.index[(trade == 1) & (px["position"] == 0)]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(px["Close"], label=f"{SYMBOL} close", linewidth=1.1)
ax.plot(px["sma_fast"], label=f"{FAST}-day average")
ax.plot(px["sma_slow"], label=f"{SLOW}-day average")
ax.scatter(entries, px.loc[entries, "Open"], marker="^", s=70, color="#008300",
           edgecolors="white", linewidths=1, zorder=3, label="Buy (next open)")
ax.scatter(exits, px.loc[exits, "Open"], marker="v", s=70, color="#e34948",
           edgecolors="white", linewidths=1, zorder=3, label="Sell (next open)")
ax.set_title(f"{SYMBOL}: {FAST}/{SLOW}-day moving-average crossover")
ax.set_xlabel("Date")
ax.set_ylabel("Price (adjusted)")
ax.legend(loc="upper left", ncol=2)
plt.tight_layout()

## 6. Equity curves and drawdowns

The top panel compounds the daily returns of each approach into the value of $10,000; the bottom panel shows each curve's drawdown from its own previous peak. Trend following usually gives up some return in steady bull markets in exchange for side-stepping part of the big declines, and the drawdown panel is where that trade-off shows.

In [ ]:
equity = START_VALUE * (1 + px[["strategy", "buy_hold"]]).cumprod()
equity.columns = ["SMA crossover", "Buy & hold"]
drawdown = (equity / equity.cummax() - 1) * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True, height_ratios=[2, 1])
for name in equity.columns:
    ax1.plot(equity[name], label=f"{name}: ${equity[name].iloc[-1]:,.0f}")
    ax2.plot(drawdown[name], label=name, linewidth=1.1)
ax1.set_title(f"{SYMBOL}: growth of ${START_VALUE:,.0f} after {COST_BPS} bp per trade")
ax1.set_ylabel("Value ($)")
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax1.legend(loc="upper left")
ax2.set_title("Drawdown from previous peak")
ax2.set_ylabel("Drawdown (%)")
ax2.set_xlabel("Date")
ax2.legend(loc="lower left")
plt.tight_layout()

## 7. Statistics side by side

- **Total return** and **CAGR**: the final value vs the start, in total and per year.
- **Volatility** and **Sharpe**: annualised with 252 days; Sharpe uses a risk-free rate of 0. Days in cash count as zero-return days, which is why the strategy's volatility is lower.
- **Max drawdown**: the worst peak-to-trough fall of the equity curve.
- **Trades**: number of entries (round trips); buy & hold buys once.
- **Exposure**: the share of days with money in the market.

In [ ]:
def stats(r, position):
    growth = (1 + r).cumprod()
    years = (r.index[-1] - r.index[0]).days / 365.25
    return {
        "Total return (%)": (growth.iloc[-1] - 1) * 100,
        "CAGR (%)": (growth.iloc[-1] ** (1 / years) - 1) * 100,
        "Volatility (%)": r.std() * np.sqrt(252) * 100,
        "Sharpe (rf = 0)": r.mean() / r.std() * np.sqrt(252),
        "Max drawdown (%)": (growth / growth.cummax().clip(lower=1) - 1).min() * 100,
        "Trades": int((position.diff().fillna(position.iloc[0]) == 1).sum()),
        "Exposure (% of days)": position.mean() * 100,
    }

always_in = pd.Series(1, index=px.index)
table = pd.DataFrame({
    "SMA crossover": stats(px["strategy"], px["position"]),
    "Buy & hold": stats(px["buy_hold"], always_in),
}).T
table["Trades"] = table["Trades"].astype(int)
table.index.name = "Approach"
table.round(2)

## 8. Every trade

Each round trip from the entry open to the exit open, net of both trade costs. A trade still open at the end of the data is marked to the last close and shown as *open*. Compounding all the trade returns gives back exactly the strategy's total return, which is a useful check that the backtest adds up.

In [ ]:
rows = []
for entry in entries:
    later_exits = exits[exits > entry]
    is_open = len(later_exits) == 0
    exit_day = px.index[-1] if is_open else later_exits[0]
    exit_price = px.loc[exit_day, "Close" if is_open else "Open"]     # open trades: last close
    ret = exit_price / px.loc[entry, "Open"] * (1 - cost) ** (1 if is_open else 2) - 1
    rows.append({"Entry": entry, "Exit": exit_day, "Status": "open" if is_open else "closed",
                 "Days held": (exit_day - entry).days, "Return (%)": ret * 100})
trades = pd.DataFrame(rows, columns=["Entry", "Exit", "Status", "Days held", "Return (%)"])
trades.index = pd.RangeIndex(1, len(trades) + 1, name="Trade")

if trades.empty:
    print("No crossovers in this period - try a longer PERIOD or shorter averages.")
else:
    compounded = ((1 + trades["Return (%)"] / 100).prod() - 1) * 100
    print(f"{len(trades)} trades, {(trades['Return (%)'] > 0).mean():.0%} winners, "
          f"average {trades['Return (%)'].mean():+.2f}% per trade")
    print(f"Compounded trade returns {compounded:.2f}% vs strategy total "
          f"{table.loc['SMA crossover', 'Total return (%)']:.2f}%")
trades.round(2)

## Try next

- Try other markets: `QQQ`, `GLD`, `TLT` or `BTC-USD`. Trend following tends to shine in long, one-directional moves.
- Change `FAST` and `SLOW` (20/100, 10/50) and `COST_BPS` (20, 50). Be careful: picking the best pair after seeing the results is curve fitting, and it will not repeat.
- One symbol over one period is an anecdote, not evidence. A robust test checks many markets and periods and keeps some data aside for out-of-sample testing.

---
*— Fincept Notebook · part of Fincept Terminal. Edit any cell and press Ctrl+Enter to run.*